# Übungen – Block 12: Praxisprojekt

Dieses Notebook beschreibt das Abschlussprojekt. Die konkreten Hostnamen und Dienste werden an die Schulungsumgebung angepasst.

## Musterlösung

Die Lösung ist bewusst als Referenzarchitektur formuliert. Es sind mehrere gleichwertige Projektvarianten möglich.

## Projektauftrag

Automatisieren Sie eine kleine Linux-Serverlandschaft mit mindestens zwei Hostgruppen.

Der Gesamtprozess soll Linux, Python und Ansible sinnvoll kombinieren.

## Teil 1: Architektur und Verantwortlichkeiten

Dokumentieren Sie:

- Control Node
- Managed Nodes
- Hostgruppen
- SSH-Zugriff
- welche Aufgabe Linux, Shell, Python und Ansible jeweils übernehmen.

```text
Control Node: admin01
Managed Nodes: web01, web02, db01
Gruppen: web, db

Shell: optionaler Startablauf
Python: Datenvalidierung und Ableitung
Ansible: Paket, Datei, Dienst, Konfiguration
```

## Teil 2: Python-Komponente

Erstellen Sie ein Python-Programm, das eine JSON-Datei mit Server- oder Anwendungsdaten einliest, Pflichtfelder prüft und daraus eine neue JSON-Datei mit abgeleiteten Konfigurationswerten erzeugt.

```python
import json
from pathlib import Path

src = Path("servers.json")
dst = Path("generated.json")

servers = json.loads(src.read_text(encoding="utf-8"))
result = []

for server in servers:
    if "name" not in server or "role" not in server:
        raise ValueError("name und role sind Pflichtfelder")

    item = dict(server)
    item["service_port"] = 8080 if server["role"] == "web" else 5432
    result.append(item)

dst.write_text(json.dumps(result, indent=2), encoding="utf-8")
```

## Teil 3: Ansible-Projekt

Erstellen Sie:

- Inventory
- `group_vars` bzw. `host_vars`
- mindestens eine Rolle
- Template
- Handler
- mindestens einen factabhängigen Task

Die Rolle soll einen Dienst bzw. ein freigegebenes Paket verwalten und eine Konfigurationsdatei erzeugen.

```text
ansible/
├── inventory.yml
├── group_vars/
│   ├── web.yml
│   └── db.yml
├── host_vars/
├── roles/
│   └── app_config/
│       ├── defaults/main.yml
│       ├── handlers/main.yml
│       ├── tasks/main.yml
│       └── templates/app.conf.j2
└── site.yml
```

`site.yml`:

```yaml
- hosts: all
  become: true
  roles:
    - app_config
```

## Teil 4: Integration

Binden Sie die vom Python-Programm erzeugten Daten in den Gesamtprozess ein. Entscheiden Sie, ob ein einfaches Shell-Skript die Aufrufe nacheinander orchestrieren soll.

Optionales Shell-Skript:

```bash
#!/bin/bash
set -e

python3 generate_config.py
ansible all -i ansible/inventory.yml -m ping
ansible-playbook -i ansible/inventory.yml ansible/site.yml
```

Die Shell koordiniert nur vorhandene Werkzeuge; die Programmlogik bleibt in Python, die Systemzustände in Ansible.

## Teil 5: Test

1. Führen Sie Connectivity Checks aus.
2. Starten Sie den ersten vollständigen Run.
3. Prüfen Sie Zielzustände.
4. Führen Sie denselben Run erneut aus.
5. Dokumentieren Sie alle noch auftretenden `changed`-Tasks.
6. Erzeugen Sie einen kontrollierten Fehlerfall und analysieren Sie ihn.

Erwartetes Testvorgehen:

```bash
ansible all -i ansible/inventory.yml -m ping
ansible-playbook -i ansible/inventory.yml ansible/site.yml
ansible-playbook -i ansible/inventory.yml ansible/site.yml
```

Der zweite Run sollte bei unverändertem Zielzustand möglichst keine Änderungen mehr melden.

Für den Fehlerfall kann beispielsweise ein falscher Hostname oder eine ungültige Variable bewusst gesetzt und anschließend mit `-vv` sowie `debug` untersucht werden.

## Abschlussreflexion

Beantworten Sie:

1. Welche Aufgaben waren mit Shell sinnvoll?
2. Welche Logik gehörte nach Python?
3. Welche Aufgaben waren typische Ansible-Zustandsverwaltung?
4. Wo spielte Idempotenz eine Rolle?
5. Wie würde sich die Lösung bei 100 Hosts verändern?
6. Welche Teile könnten als nächstes professionalisiert werden?

## Musterantwort zur Abschlussreflexion

1. Shell eignet sich für einen einfachen Aufrufablauf mehrerer vorhandener Programme.
2. Validierung, Datenverarbeitung und eigene Berechnungslogik gehören in Python.
3. Paketinstallation, Dateiverteilung, Konfiguration und Dienstzustände sind typische Ansible-Aufgaben.
4. Idempotenz zeigt sich vor allem beim wiederholten Ansible-Run.
5. Bei 100 Hosts bleiben Rollen und Inventories grundsätzlich tragfähig; Gruppenstruktur, Fehlermanagement, Parallelisierung und sichere Secrets-Verwaltung gewinnen an Bedeutung.
6. Nächste Professionalisierungsschritte wären beispielsweise Vault/Secrets, Collections, CI-Tests, Linting, Molecule oder zentrale Automation-Plattformen.
